[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/05-sequences-before-transformers.ipynb)

# Module 5: Sequences Before Transformers

Companion notebook to [Module 5](https://transformers-and-math.drmhse.com/modules/05-sequences-before-transformers) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Part 1 — An RNN, one step at a time (section 5.2)

In [ ]:
d = 32
cell = nn.RNNCell(input_size=8, hidden_size=d, nonlinearity='tanh')
xs = torch.randn(60, 1, 8)
h = torch.zeros(1, d)
hs = []
for x in xs:            # steps must run in order: each needs the previous h
    h = cell(x, h)
    hs.append(h)
print('hidden state shape at every step:', tuple(h.shape))

## Part 2 — How much gradient reaches early steps (section 5.4)
Scale $W_{hh}$ so its largest stretch $\sigma_{\max}$ takes several values, and measure $\|\partial \mathbf{h}_T / \partial \mathbf{h}_t\|$ with autograd.

In [ ]:
def gradient_by_distance(sigma, T=60):
    torch.manual_seed(1)
    cell = nn.RNNCell(8, d, nonlinearity='tanh')
    with torch.no_grad():
        W = cell.weight_hh
        W *= sigma / torch.linalg.matrix_norm(W, ord=2)
    xs = torch.randn(T, 1, 8)
    h0 = torch.zeros(1, d, requires_grad=True)
    h, states = h0, []
    for x in xs:
        h = cell(x, h)
        h.retain_grad()
        states.append(h)
    states[-1].sum().backward()
    return [states[t].grad.norm().item() for t in range(T)]

for sigma in (0.9, 1.0, 1.5):
    g = gradient_by_distance(sigma)
    plt.semilogy(list(range(len(g)))[::-1], g, label=f'σ_max = {sigma}')
plt.xlabel('distance back from the last step'); plt.ylabel('gradient size'); plt.legend(); plt.show()

## Part 3 — Bahdanau attention (section 5.6)
Instead of one summary vector, the decoder scores every encoder state, softmaxes the scores and reads a weighted average: the context vector.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.Ws, self.Wh, self.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, 1)
    def forward(self, s, H):                   # s: (d,), H: (M, d)
        e = self.v(torch.tanh(self.Ws(s) + self.Wh(H))).squeeze(-1)   # (M,) scores
        alpha = torch.softmax(e, dim=0)
        return alpha @ H, alpha                 # context vector, weights

H = torch.stack(hs).squeeze(1)[:10]             # ten encoder states
ctx, alpha = AdditiveAttention(d)(torch.randn(d), H)
print('weights sum to', alpha.sum().item(), '; context shape', tuple(ctx.shape))

## Your turn
1. In Part 2, what happens at σ_max = 3? Why can a *stretching* matrix still give vanishing gradients? (Look at the tanh slopes.)
2. Time the loop in Part 1 for 1,000 steps, and compare with one matrix multiplication of a (1000 × 8) input by an (8 × 32) matrix.

<details><summary>Hint</summary>

1. Large pre-activations saturate tanh, so $1 - \tanh^2$ is near zero and the product shrinks anyway.
</details>